In [1]:
import cv2
import mediapipe as mp
import time
import math

In [2]:
BaseOptions = mp.tasks.BaseOptions
FaceLandmarker = mp.tasks.vision.FaceLandmarker
FaceLandmarkerOptions = mp.tasks.vision.FaceLandmarkerOptions
VisionRunningMode = mp.tasks.vision.RunningMode

In [8]:
LEFT_EYE = [33, 160, 158, 133, 153, 144]
RIGHT_EYE = [362, 385, 387, 263, 373, 380]

EAR_THRESHOLD = 0.25

right_blink_count = 0
left_blink_count = 0
right_eye_closed = False
left_eye_closed = False
instruction = "RIGHT"

model_path = "face_landmarker.task"


In [6]:
def distance(p1, p2):
    return math.dist([p1.x, p1.y], [p2.x, p2.y])

def eye_aspect_ratio(landmark, eye_indices):
    p1, p2, p3, p4, p5, p6 = [landmark[i] for i in eye_indices]
    vertical1 = distance(p2, p6)
    vertical2 = distance(p3, p5)
    horizontal = distance(p1, p4)
    return (vertical1 + vertical2) / (2.0 * horizontal)

In [11]:
def print_result(result, output_image, timestamp_ms):
    global right_blink_count, left_blink_count
    global right_eye_closed, left_eye_closed
    global instruction

    if not result.face_landmarks:
        return
    
    landmarks = result.face_landmarks[0]

    left_ear = eye_aspect_ratio(landmarks, LEFT_EYE)
    right_ear = eye_aspect_ratio(landmarks, RIGHT_EYE)

    left_closed = left_ear < EAR_THRESHOLD
    right_closed = right_ear < EAR_THRESHOLD

    if instruction == "RIGHT":
        if right_closed and not left_closed and not right_eye_closed:
            right_eye_closed = True

        elif not right_closed and right_eye_closed and not left_closed:
            right_blink_count += 1
            right_eye_closed = False
            instruction = "LEFT"
            print("Right eye blink detected")

    elif instruction == 'LEFT':
        if left_closed and not right_closed and not left_eye_closed:
            left_eye_closed = True

        elif not left_closed and left_eye_closed and not right_closed:
            left_blink_count += 1
            left_eye_closed = False
            instruction = "RIGHT"
            print("Left eye blink detected")

    if left_closed and right_closed:
        left_eye_closed = False
        right_eye_closed = False

In [12]:
options = FaceLandmarkerOptions(
    base_options = BaseOptions(model_asset_path=model_path),
    running_mode = VisionRunningMode.LIVE_STREAM,
    result_callback=print_result
)

In [13]:
with FaceLandmarker.create_from_options(options) as landmarker:
    cap = cv2.VideoCapture(0)

    while cap.isOpened():
        success, frame = cap.read()
        if not success:
            continue

        frame = cv2.flip(frame, 1)
        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

        mp_image = mp.Image(
            image_format=mp.ImageFormat.SRGB,
            data = frame_rgb
        )

        timestamp_ms = int(time.time() * 1000)
        landmarker.detect_async(mp_image, timestamp_ms)

        cv2.putText(frame, f"Blink {instruction} eye", (30, 40),
                    cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 255), 2)

        cv2.putText(frame, f"Right: {right_blink_count}", (30, 80),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)

        cv2.putText(frame, f"Left: {left_blink_count}", (30, 120),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)

        cv2.imshow("Assignment 9 - Blink Logic", frame)

        if cv2.waitKey(1) & 0xFF == ord('q'):
            break
    
    cap.release()
    cv2.destroyAllWindows()

Right eye blink detected
Left eye blink detected
Right eye blink detected
Left eye blink detected
Right eye blink detected
Left eye blink detected
Right eye blink detected
Left eye blink detected
